#Ingestion de archivo "Movie.csv"



###Paso 1 - Leer el archivo CSV usando "DataFrameReader" de Spark

In [0]:
#vemos el contenido del contenedor Bronze
display(dbutils.fs.ls("abfss://bronze@lsdata01.dfs.core.windows.net/"))

In [0]:
#Vemnos el contenido del archivo movie.
# inferSchema, el esquema lo infiere el propio DataBricks
movie_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/movie.csv")

In [0]:
display(movie_df)

In [0]:
#Vemo que todas las columnas esta definidas como stream
movie_df.printSchema()

In [0]:
# Vemos los datos promedios de cada columna.
display(movie_df.describe())

### En el siguiente caso, definimos el tipo de dato con StructField

In [0]:
#Importamos los tipos de datos que vamos a necesitar

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

In [0]:
# StructType, definimos el esquema para cada tipo de dato
movie_schema = StructType ( fields = [
    StructField("movieId", IntegerType(), False),
    StructField("title", StringType(), True),
    StructField("budget", DoubleType(), True),
    StructField("homePage", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("yearReleaseDate", IntegerType(), True),
    StructField("releaseDate", DateType(), True),
    StructField("revenue", DoubleType(), True),
    StructField("durationTime", IntegerType(), True),
    StructField("movieStatus", StringType(), True),
    StructField("tagline", StringType(), True),
    StructField("voteAverage", DoubleType(), True),
    StructField("voteCount", IntegerType(), True)
] )



In [0]:
movie_df = spark.read \
    .option("header", True) \
    .schema(movie_schema) \
    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/movie.csv")

In [0]:
#Vemo que todas las columnas esta definidas como stream
movie_df.printSchema()

##Paso 2 - Seleccionar las columnas que se requieren

In [0]:
#usamos el col, para llamar a las columnas y de paso utilizar funciones

from pyspark.sql.functions import col


In [0]:
movies_selected_df = movie_df.select(col("movieId"), col("title"), col("budget"), col("popularity"), col("yearReleaseDate"), col("releaseDate"), col("revenue"), col("durationTime"), col("voteAverage"), col("voteCount").alias("vote_count"))



In [0]:
display(movies_selected_df)

##Paso 3 -Renombrar columnas

In [0]:
movies_renamed_df = movies_selected_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("yearReleaseDate", "year_release_date")\
    .withColumnRenamed("releaseDate", "release_date")\
    .withColumnRenamed("durationTime", "duration_time")\
    .withColumnRenamed("voteAverage", "vote_average")

In [0]:
display(movies_renamed_df)

##Paso 4 - Añadir columnas a una tabla

In [0]:
from pyspark.sql.functions import  current_timestamp

In [0]:
movie_final_df = movies_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
display(movie_final_df)

In [0]:
from pyspark.sql.functions import lit

movie_final_df = movies_renamed_df\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("env", lit("produccion"))


display(movie_final_df)


In [0]:

movie_final_2_df = movie_final_df.withColumns({"ingestion_date_2": current_timestamp(), "env_2": lit("produccion")})

display(movie_final_2_df)


##Paso 5 - Guardar datos en datalake en formato parket

In [0]:
movie_final_df.write.parquet("abfss://silver@lsdata01.dfs.core.windows.net/movies")

In [0]:
%fs
ls abfss://silver@lsdata01.dfs.core.windows.net/movies

In [0]:
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/movies")
display(df)

In [0]:
movie_final_df.write.mode("overwrite").parquet("abfss://silver@lsdata01.dfs.core.windows.net/movies")

In [0]:
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/movies")
display(df)

## Paso 5.1 - Guardar en formato parquet pero particionando por el campo year_release_date

In [0]:
movie_final_df.write.mode("overwrite").partitionBy("year_release_date").format("parquet").save("abfss://silver@lsdata01.dfs.core.windows.net/movies")


In [0]:
%fs
ls abfss://silver@lsdata01.dfs.core.windows.net/movies